In [ ]:
!git clone https://github.com/aimacode/aima-python.git

%cd aima-python
!pip install -r requirements.txt

Cloning into 'aima-python'...
remote: Enumerating objects: 5095, done.
remote: Total 5095 (delta 0), reused 0 (delta 0), pack-reused 5095 (from 1)
Receiving objects: 100% (5095/5095), 17.44 MiB | 18.68 MiB/s, done.
Resolving deltas: 100% (3418/3418), done.
/content/aima-python
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.9/92.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.1/254.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 4.1 MB/s eta 0:00:00
  Created wheel for image: filename=image-1.5.33-py2.py3-none-any.whl size=19482 sha256=6261029237dc3e39f97542802945975da74b71dc6d019

In [ ]:
import sys
from search import Problem, astar_search
from utils import *

In [ ]:
""""
PROBLEMA:
jogo Pipe Mania

Descrição: O agente opera em um grid NxN, o agente é o sistema que deve rotacionar
peças de cano espalhadas pelo mapa, cada célula do grid possui um tipo de cano
(reto, curva em L, cruzamento) ou está vazia. Há uma origem de água e um destino
e o objetivo é realizar o menor número de rotações nas peças de cano para criar
um caminho fechado e contínuo da origem ao destino, garantindo que não haja vazamentos para as bordas.

"""
class ProblemaEncanamento(Problem):
    def __init__(self, matriz_inicial, pos_inicio, pos_fim):
        """
        Inicializa o problema recebendo o mapa original, o ponto de partida e o destino.
        A função é converter a matriz em uma estrutura de tuplas imutáveis, adicionando um
        marcador de em cada célula para que o algoritmo A* possa memorizar os
        estados do tabuleiro. Todas as peças nascem False para forçar o agente a testar tudo.
        I, F e X nascem fixadas por serem imutáveis.

        """
        estado_imutavel = tuple(
            tuple((celula['tipo'], celula['rot'], celula['tipo'] in ['I', 'F', 'X'])
            for celula in linha) for linha in matriz_inicial
        )

        self.initial = estado_imutavel
        self.pos_inicio = pos_inicio
        self.pos_fim = pos_fim
        self.tamanho = len(matriz_inicial)

        super().__init__(self.initial)

    def _pegar_mascara(self, tipo, rot):
        """
        Recebe o tipo de cano e sua rotação atual e devolve um valor numérico único (uma máscara de bits).
        Esse valor representa para quais direções aquele cano possui aberturas.


        Retorna o valor da máscara de bits para uma peça e sua rotação.
        Norte=1, Leste=2, Sul=4, Oeste=8 (potência de 2, 2^0, 2^1, 2^2 e 2^3). Cada soma gera um valor único o que garante
        a certeza das posições a partir dos valores de cada máscara.

        O cano reto só possui duas posições válidas, vertical e horizontal, calcular 4 possíveis posições é desnecessário.
        Já o cano L precisa ter as 4 posições calculadas pois cada posição é única.
        O cano cruz não é rotacionado por ter entrada para todas as direções.

        """
        mascaras = {
            'R': [5, 10, 5, 10],      # 0/2: Norte e Sul (1+4=5), 1/3: Leste e Oeste (2+8=10)
            'L': [3, 6, 12, 9],       # 0:Norte e Leste (3), 1: Leste e Sul(6), 2: Sul e Oeste(12), 3:Oeste e Norte(9)
            'C': [15, 15, 15, 15],    # Cruzamento sempre tem as 4 saídas
            'I': [1, 2, 4, 8],        # Início só tem 1 saída
            'F': [1, 2, 4, 8],        # Fim só tem 1 entrada
            'X': [0, 0, 0, 0]         # NULL não tem conexões
        }
        return mascaras[tipo][rot]

    def _tracar_agua(self, state):
        """
        Calcula a trajetória a partir do Início, percorrendo os canos conectados
        até esbarrar em uma borda, espaço vazio, destino ou em um cano ainda não avaliado.
        Ele devolve a coordenada exata de onde o fluxo travou e a direção de entrada necessária.

        Simula o fluxo da água a partir do Início.
        Retorna onde a água parou (x, y) e qual direção ela quer ir.

        A matriz começa de cima para baixo e da esquerda para a direita. Canto superior esquerdo sendo (0, 0).

        """
        movimentos = {
            1: (-1, 0, 1, 4), # Norte precisa conectar com Sul (4)
            2: (0, 1, 2, 8),  # Leste precisa conectar com Oeste (8)
            4: (1, 0, 4, 1),  # Sul precisa conectar com Norte (1)
            8: (0, -1, 8, 2)  # Oeste precisa conectar com Leste (2)
        }

        linha_atual, col_atual = self.pos_inicio
        tipo_atual, rot_atual, _ = state[linha_atual][col_atual]

        dir_fluxo = self._pegar_mascara(tipo_atual, rot_atual)

        while True:
            dl, dc, mask_saida, mask_entrada = movimentos[dir_fluxo]

            nova_linha = linha_atual + dl
            nova_col = col_atual + dc

            if not (0 <= nova_linha < self.tamanho and 0 <= nova_col < self.tamanho):
                return (linha_atual, col_atual), (nova_linha, nova_col), mask_entrada, False

            # Atualiza para desempacotar o status 'fixed_vizinho' - tava crashando o código sem isso
            tipo_vizinho, rot_vizinho, fixed_vizinho = state[nova_linha][nova_col]

            if tipo_vizinho == 'X':
                return (linha_atual, col_atual), (nova_linha, nova_col), mask_entrada, False

            # Se a peça ainda não foi avaliada pelo agente, a água para
            # para dar a chance do agente girá-la e testar os caminhos.
            if not fixed_vizinho:
                return (linha_atual, col_atual), (nova_linha, nova_col), mask_entrada, False

            mascara_vizinho = self._pegar_mascara(tipo_vizinho, rot_vizinho)

            # Se a peça já foi fixada pelo agente e conecta, a água avança
            if (mascara_vizinho & mask_entrada) > 0:
                linha_atual, col_atual = nova_linha, nova_col

                if (linha_atual, col_atual) == self.pos_fim:
                    return (linha_atual, col_atual), None, None, True

                # Descobre para onde a água vai sair agora
                if tipo_vizinho == 'C':
                    # No cruzamento a água passa reto, então a direção do fluxo se mantém a mesma
                    pass
                else:
                    # Nas peças R e L, a subtração funciona perfeitamente
                    dir_fluxo = mascara_vizinho - mask_entrada
            else:
                return (linha_atual, col_atual), (nova_linha, nova_col), mask_entrada, False

    def actions(self, state):
        """

        Em vez de tentar girar qualquer peça do mapa de forma aleatória, a função usa a simulação da água para focar apenas no cano atual que precisa de rotação.
        Ela testa as rotações localmente e devolve apenas as ações que fazem a água percorrer pelo cano que não está conectado.

        Retorna as rotações possíveis para a peça que está imediatamente na frente do fluxo de água.

        """
        acoes_possiveis = []
        pos_atual, pos_vizinho, mask_entrada_necessaria, chegou_no_fim = self._tracar_agua(state)

        if chegou_no_fim or pos_vizinho is None:
            return acoes_possiveis

        l_vizinho, c_vizinho = pos_vizinho

        if 0 <= l_vizinho < self.tamanho and 0 <= c_vizinho < self.tamanho:
            tipo_vizinho, rot_atual, fixed_vizinho = state[l_vizinho][c_vizinho]

            if tipo_vizinho not in ['I', 'F', 'X']:

                rotacoes_para_testar = []
                if tipo_vizinho == 'C':
                    rotacoes_para_testar = [0]
                elif tipo_vizinho == 'R':
                    rotacoes_para_testar = [0, 1]
                elif tipo_vizinho == 'L':
                    rotacoes_para_testar = [0, 1, 2, 3]

                for nova_rot in rotacoes_para_testar:
                    mascara_teste = self._pegar_mascara(tipo_vizinho, nova_rot)

                    if (mascara_teste & mask_entrada_necessaria) > 0:
                        acao = ('rotacionar', (l_vizinho, c_vizinho), nova_rot)
                        acoes_possiveis.append(acao)

        return acoes_possiveis

    def result(self, state, action):
        """
        Recebe o estado do tabuleiro atual e a ação escolhida pelo agente, no caso, girar uma peça específica.
        O método cria uma cópia inteiramente nova do mapa com a peça já rotacionada e com o status de avaliada,
        devolvendo esse novo tabuleiro ao A*.

        """
        _, (linha_alvo, col_alvo), nova_rotacao = action

        novo_estado = [list(linha) for linha in state]

        tipo_peca = novo_estado[linha_alvo][col_alvo][0]

        # O agente tomou a decisão e a peça ganha a nova rotação de acordo com as posições e o True para ser fixada (consequentemente não pode mais se mexida)
        novo_estado[linha_alvo][col_alvo] = (tipo_peca, nova_rotacao, True)

        return tuple(tuple(linha) for linha in novo_estado)

    def goal_test(self, state):
        """
        Sempre que o algoritmo analisa um novo estado do tabuleiro gerado, essa função utiliza a simulação da água para verificar se o fluxo contínuo
        conseguiu chegar à coordenada de destino, caso a busca tenha sucesso, encerra a busca.

        Retorna True se o problema foi resolvido.

        """
        _, _, _, chegou_no_fim = self._tracar_agua(state)
        return chegou_no_fim

    def h(self, node):
        """
        É a Heurística que torna o agente inteligente em vez de aleatório. A função verifica onde a ponta da água está travada e calcula a
        Distância de Manhattan (indispensável no uso do A* como visto em aula) desse ponto até o destino final. Isso permite ao algoritmo A* priorizar
        matematicamente os caminhos que parecem mais curtos.

        """
        state = node.state
        pos_atual, _, _, chegou_no_fim = self._tracar_agua(state)

        if chegou_no_fim:
            return 0

        linha_atual, col_atual = pos_atual
        linha_fim, col_fim = self.pos_fim

        distancia = abs(linha_atual - linha_fim) + abs(col_atual - col_fim)

        return distancia

In [ ]:
# CENÁRIO 1: FÁCIL
matriz_facil = [
    [{'tipo': 'I', 'rot': 1}, {'tipo': 'L', 'rot': 0}, {'tipo': 'X', 'rot': 0}],
    [{'tipo': 'X', 'rot': 0}, {'tipo': 'L', 'rot': 1}, {'tipo': 'L', 'rot': 3}],
    [{'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'F', 'rot': 0}]
]
posicao_inicio = (0, 0)
posicao_fim = (2, 2)

# CENÁRIO 2: DIFÍCIL (Labirinto com Armadilhas)
matriz_dificil = [
    [{'tipo': 'I', 'rot': 1}, {'tipo': 'R', 'rot': 1}, {'tipo': 'R', 'rot': 0}, {'tipo': 'L', 'rot': 3}, {'tipo': 'L', 'rot': 0}],
    [{'tipo': 'R', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'R', 'rot': 1}, {'tipo': 'X', 'rot': 0}],
    [{'tipo': 'L', 'rot': 2}, {'tipo': 'L', 'rot': 1}, {'tipo': 'R', 'rot': 0}, {'tipo': 'L', 'rot': 2}, {'tipo': 'X', 'rot': 0}],
    [{'tipo': 'X', 'rot': 0}, {'tipo': 'R', 'rot': 1}, {'tipo': 'X', 'rot': 0}, {'tipo': 'C', 'rot': 0}, {'tipo': 'X', 'rot': 0}],
    [{'tipo': 'X', 'rot': 0}, {'tipo': 'L', 'rot': 3}, {'tipo': 'R', 'rot': 0}, {'tipo': 'R', 'rot': 1}, {'tipo': 'F', 'rot': 3}]
]
pos_inicio_dificil = (0, 0)
pos_fim_dificil = (4, 4)

# CENÁRIO 3: IMPOSSÍVEL (Sem Solução)
matriz_impossivel = [
    [{'tipo': 'I', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'R', 'rot': 1}, {'tipo': 'L', 'rot': 2}],
    [{'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'R', 'rot': 0}, {'tipo': 'L', 'rot': 3}],
    [{'tipo': 'R', 'rot': 1}, {'tipo': 'R', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}],
    [{'tipo': 'L', 'rot': 3}, {'tipo': 'L', 'rot': 1}, {'tipo': 'X', 'rot': 0}, {'tipo': 'F', 'rot': 0}]
]
pos_inicio_impossivel = (0, 0)
pos_fim_impossivel = (3, 3)


# CENÁRIO 4: GIGANTE, PORÉM FÁCIL (8x8)
matriz_gigante = [
    [{'tipo': 'I', 'rot': 2}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}],
    [{'tipo': 'R', 'rot': 1}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}],
    [{'tipo': 'R', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}],
    [{'tipo': 'R', 'rot': 1}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}],
    [{'tipo': 'R', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}],
    [{'tipo': 'R', 'rot': 1}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}],
    [{'tipo': 'R', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}],
    [{'tipo': 'L', 'rot': 2}, {'tipo': 'R', 'rot': 1}, {'tipo': 'R', 'rot': 0}, {'tipo': 'R', 'rot': 1}, {'tipo': 'R', 'rot': 1}, {'tipo': 'R', 'rot': 0}, {'tipo': 'R', 'rot': 1}, {'tipo': 'F', 'rot': 3}]
]
pos_inicio_gigante = (0, 0)
pos_fim_gigante = (7, 7)

# CENÁRIO 5: GIGANTE E DIFÍCIL (8x8 com Labirinto e Múltiplas Rotas)

matriz_gigante_dificil = [
    [{'tipo': 'I', 'rot': 2}, {'tipo': 'X', 'rot': 0}, {'tipo': 'R', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'L', 'rot': 1}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}],
    [{'tipo': 'L', 'rot': 2}, {'tipo': 'R', 'rot': 0}, {'tipo': 'C', 'rot': 0}, {'tipo': 'L', 'rot': 0}, {'tipo': 'R', 'rot': 1}, {'tipo': 'L', 'rot': 3}, {'tipo': 'X', 'rot': 0}, {'tipo': 'R', 'rot': 0}],
    [{'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'R', 'rot': 1}, {'tipo': 'R', 'rot': 1}, {'tipo': 'X', 'rot': 0}, {'tipo': 'L', 'rot': 2}, {'tipo': 'R', 'rot': 0}, {'tipo': 'X', 'rot': 0}],
    [{'tipo': 'L', 'rot': 1}, {'tipo': 'X', 'rot': 0}, {'tipo': 'L', 'rot': 0}, {'tipo': 'C', 'rot': 0}, {'tipo': 'R', 'rot': 0}, {'tipo': 'L', 'rot': 2}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}],
    [{'tipo': 'R', 'rot': 1}, {'tipo': 'L', 'rot': 3}, {'tipo': 'R', 'rot': 0}, {'tipo': 'L', 'rot': 1}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}],
    [{'tipo': 'X', 'rot': 0}, {'tipo': 'R', 'rot': 1}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'L', 'rot': 1}, {'tipo': 'R', 'rot': 1}, {'tipo': 'L', 'rot': 0}, {'tipo': 'X', 'rot': 0}],
    [{'tipo': 'X', 'rot': 0}, {'tipo': 'L', 'rot': 2}, {'tipo': 'R', 'rot': 0}, {'tipo': 'C', 'rot': 0}, {'tipo': 'R', 'rot': 0}, {'tipo': 'R', 'rot': 0}, {'tipo': 'L', 'rot': 1}, {'tipo': 'X', 'rot': 0}],
    [{'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'R', 'rot': 1}, {'tipo': 'X', 'rot': 0}, {'tipo': 'X', 'rot': 0}, {'tipo': 'L', 'rot': 2}, {'tipo': 'F', 'rot': 3}]
]
pos_inicio_gig_dif = (0, 0)
pos_fim_gig_dif = (7, 7)

In [ ]:
# Opção 1: Fácil
# meu_projeto = ProblemaEncanamento(matriz_facil, posicao_inicio, posicao_fim)

# Opção 2: Difícil
# meu_projeto = ProblemaEncanamento(matriz_dificil, pos_inicio_dificil, pos_fim_dificil)

# Opção 3: Impossível
# meu_projeto = ProblemaEncanamento(matriz_impossivel, pos_inicio_impossivel, pos_fim_impossivel)

# Opção 4: Gigante e fácil
# meu_projeto = ProblemaEncanamento(matriz_gigante, pos_inicio_gigante, pos_fim_gigante)

#Opção 5: Gigante e difícil
meu_projeto = ProblemaEncanamento(matriz_gigante_dificil, pos_inicio_gig_dif, pos_fim_gig_dif)

print("Iniciando a busca")
resultado = astar_search(meu_projeto)

if resultado is None:
    # Se nenhum caminho for válido
    print("\nNenhuma solução possível.")
    print("-1")
else:
    print("\nSolução: ")

    # .path() retorna todos os nós desde a origem até o fim
    caminho = resultado.path()

    # Pulamos o primeiro nó, pois é o estado inicial, porque ele não teve ação
    for no in caminho[1:]:
        acao_tomada = no.action

        # O formato da ação que criamos foi: ('rotacionar', (linha, coluna), nova_rotacao)
        _, coordenada, nova_rotacao = acao_tomada
        x, y = coordenada

        print(f"Peça em ({x}, {y}) rotacionada para {nova_rotacao}")

    print("\nCusto total (número de rotações):", resultado.path_cost)

Iniciando a busca

Solução: 
Peça em (1, 0) rotacionada para 0
Peça em (1, 1) rotacionada para 1
Peça em (1, 2) rotacionada para 0
Peça em (1, 3) rotacionada para 2
Peça em (2, 3) rotacionada para 0
Peça em (3, 3) rotacionada para 0
Peça em (4, 3) rotacionada para 3
Peça em (4, 2) rotacionada para 1
Peça em (4, 1) rotacionada para 1
Peça em (5, 1) rotacionada para 0
Peça em (6, 1) rotacionada para 0
Peça em (6, 2) rotacionada para 1
Peça em (6, 3) rotacionada para 0
Peça em (6, 4) rotacionada para 1
Peça em (6, 5) rotacionada para 1
Peça em (6, 6) rotacionada para 2
Peça em (7, 6) rotacionada para 0

Custo total (número de rotações): 17
